In [1]:
def main(datasources, start_date, end_date):
    import time
    import numpy as np
    import pandas as pd
    import dai
    import xgboost as xgb
    import structlog

    logger = structlog.get_logger()

    EXPERIMENT = "E4_FILTER"

    TRAIN_START = "2023-01-01 00:00:00"
    TRAIN_END = "2023-12-31 23:59:59"

    LOOKBACK_DAYS = 90
    LABEL_HORIZON = 5

    bar1m_table = datasources["bar1m"]

    def get_feature_columns():
        momentum_features = [
            "ret_1d", "ret_3d", "ret_5d", "ret_10d", "ret_20d", "ret_60d",
            "vol_20d", "volume_ratio_20d", "intraday_ret", "close_position",
        ]

        price_action_lite = [
            "body_size",
            "upper_shadow",
            "lower_shadow",
            "close_strength",
        ]

        #lob_event_features = [
            #"volume_shock_count",
        #]

        return momentum_features + price_action_lite #+ lob_event_features

    feature_cols = get_feature_columns()

    def query_daily_bar(table_name, sd, ed, need_label=True):
        query_start = (
            pd.to_datetime(sd) - pd.Timedelta(days=LOOKBACK_DAYS)
        ).strftime("%Y-%m-%d %H:%M:%S")

        query_end = pd.to_datetime(ed)
        if need_label:
            query_end = query_end + pd.Timedelta(days=15)
        query_end = query_end.strftime("%Y-%m-%d %H:%M:%S")

        sql = f"""
        WITH minute_base AS (
            SELECT
                date,
                instrument,
                date_trunc('day', date)::DATE AS trading_day,

                open,
                high,
                low,
                close,
                volume,
                amount,

                ask_price1,
                bid_price1,

                COALESCE(bid_volume1, 0) AS bid_volume1,
                COALESCE(bid_volume2, 0) AS bid_volume2,
                COALESCE(bid_volume3, 0) AS bid_volume3,

                COALESCE(ask_volume1, 0) AS ask_volume1,
                COALESCE(ask_volume2, 0) AS ask_volume2,
                COALESCE(ask_volume3, 0) AS ask_volume3,

                (ask_price1 + bid_price1) / 2.0 AS mid_price,

                (ask_price1 - bid_price1)
                / ((ask_price1 + bid_price1) / 2.0 + 1e-8) AS relative_spread,

                (
                    COALESCE(bid_volume1, 0)
                    + COALESCE(bid_volume2, 0)
                    + COALESCE(bid_volume3, 0)
                ) AS bid_depth3,

                (
                    COALESCE(ask_volume1, 0)
                    + COALESCE(ask_volume2, 0)
                    + COALESCE(ask_volume3, 0)
                ) AS ask_depth3,

                (
                    ask_price1 * COALESCE(bid_volume1, 0)
                    + bid_price1 * COALESCE(ask_volume1, 0)
                )
                / (
                    COALESCE(bid_volume1, 0)
                    + COALESCE(ask_volume1, 0)
                    + 1e-8
                ) AS micro_price

            FROM {table_name}
            WHERE ask_price1 > 0
              AND bid_price1 > 0
              AND ask_price1 >= bid_price1
              AND date >= '{query_start}'
              AND date <= '{query_end}'
        ),

        minute_feature AS (
            SELECT
                *,

                AVG(volume) OVER (
                    PARTITION BY instrument, trading_day
                    ORDER BY date
                    ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
                ) AS minute_volume_ma20,

                (bid_depth3 - ask_depth3)
                / (bid_depth3 + ask_depth3 + 1e-8) AS obi3

            FROM minute_base
        ),

        minute_flag AS (
            SELECT
                *,

                CASE WHEN obi3 > 0.8 THEN 1 ELSE 0 END AS obi_pos_flag,
                CASE WHEN obi3 < -0.8 THEN 1 ELSE 0 END AS obi_neg_flag

            FROM minute_feature
        ),

        minute_run AS (
            SELECT
                *,

                SUM(CASE WHEN obi_pos_flag = 0 THEN 1 ELSE 0 END) OVER (
                    PARTITION BY instrument, trading_day
                    ORDER BY date
                ) AS obi_pos_group,

                SUM(CASE WHEN obi_neg_flag = 0 THEN 1 ELSE 0 END) OVER (
                    PARTITION BY instrument, trading_day
                    ORDER BY date
                ) AS obi_neg_group

            FROM minute_flag
        ),

        obi_pos_streak AS (
            SELECT
                trading_day,
                instrument,
                MAX(streak_len) AS obi_pos_persistence
            FROM (
                SELECT
                    trading_day,
                    instrument,
                    obi_pos_group,
                    COUNT(*) AS streak_len
                FROM minute_run
                WHERE obi_pos_flag = 1
                GROUP BY trading_day, instrument, obi_pos_group
            )
            GROUP BY trading_day, instrument
        ),

        obi_neg_streak AS (
            SELECT
                trading_day,
                instrument,
                MAX(streak_len) AS obi_neg_persistence
            FROM (
                SELECT
                    trading_day,
                    instrument,
                    obi_neg_group,
                    COUNT(*) AS streak_len
                FROM minute_run
                WHERE obi_neg_flag = 1
                GROUP BY trading_day, instrument, obi_neg_group
            )
            GROUP BY trading_day, instrument
        ),


        daily_bar AS (
            SELECT
                trading_day AS date,
                instrument,

                ARG_MIN(open, date) AS open,
                MAX(high) AS high,
                MIN(low) AS low,
                ARG_MAX(close, date) AS close,
                SUM(volume) AS volume,
                SUM(amount) AS amount,

                AVG(relative_spread) AS avg_spread,
                AVG(bid_depth3 + ask_depth3) AS avg_depth3,

                AVG(
                    (bid_depth3 - ask_depth3)
                    / (bid_depth3 + ask_depth3 + 1e-8)
                ) AS avg_obi3,

                AVG(
                    (micro_price - mid_price)
                    / (mid_price + 1e-8)
                ) AS avg_micro_dev,

                AVG(
                    CASE
                        WHEN relative_spread > 0
                        THEN LOG(1.0 + bid_depth3 + ask_depth3)
                             / SQRT(relative_spread + 1e-8)
                        ELSE 0
                    END
                ) AS liquidity_quality

                ,

        SUM(
            CASE
                WHEN minute_volume_ma20 > 0
                AND volume / (minute_volume_ma20 + 1e-8) > 2.0
                THEN 1
                ELSE 0
            END
        ) AS volume_shock_count

            FROM minute_run
            GROUP BY trading_day, instrument
            ORDER BY trading_day, instrument
        )

        SELECT
            d.*,
            COALESCE(p.obi_pos_persistence, 0) AS obi_pos_persistence,
            COALESCE(n.obi_neg_persistence, 0) AS obi_neg_persistence
        FROM daily_bar d
        LEFT JOIN obi_pos_streak p
        ON d.date = p.trading_day
        AND d.instrument = p.instrument
        LEFT JOIN obi_neg_streak n
        ON d.date = n.trading_day
        AND d.instrument = n.instrument
        """

        df = dai.query(sql, compression=True).df()
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
        return df

    def add_features(df, need_label=True):

        # 如果 SQL 暂时没有生成这些列，则补 0
        for col in ["volume_shock_count", "obi_pos_persistence", "obi_neg_persistence"]:
            if col not in df.columns:
                df[col] = 0

        base_num_cols = [
            "open", "high", "low", "close", "volume", "amount",
            "avg_spread", "avg_depth3", "avg_obi3",
            "avg_micro_dev", "liquidity_quality",
            "volume_shock_count","obi_pos_persistence",
            "obi_neg_persistence"
        ]

        for c in base_num_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")
            df[c] = df[c].replace([np.inf, -np.inf], np.nan)

        g = df.groupby("instrument", group_keys=False)

        # =========================
        # Momentum Features
        # =========================
        df["ret_1d"] = g["close"].pct_change(1)
        df["ret_3d"] = g["close"].pct_change(3)
        df["ret_5d"] = g["close"].pct_change(5)
        df["ret_10d"] = g["close"].pct_change(10)
        df["ret_20d"] = g["close"].pct_change(20)
        df["ret_60d"] = g["close"].pct_change(60)

        df["vol_20d"] = (
            g["ret_1d"]
            .rolling(20, min_periods=5)
            .std()
            .reset_index(level=0, drop=True)
        )

        df["volume_ma20"] = (
            g["volume"]
            .rolling(20, min_periods=5)
            .mean()
            .reset_index(level=0, drop=True)
        )

        df["volume_ratio_20d"] = df["volume"] / (df["volume_ma20"] + 1e-8)

        df["intraday_ret"] = df["close"] / (df["open"] + 1e-8) - 1

        df["close_position"] = (
            (df["close"] - df["low"])
            / (df["high"] - df["low"] + 1e-8)
        )

        # =========================
        # E2 Lite Price Action
        # =========================
        df["body_size"] = (
            (df["close"] - df["open"])
            / (df["open"] + 1e-8)
        )

        df["upper_shadow"] = (
            df["high"] - np.maximum(df["open"], df["close"])
        ) / (df["close"] + 1e-8)

        df["lower_shadow"] = (
            np.minimum(df["open"], df["close"]) - df["low"]
        ) / (df["close"] + 1e-8)

        df["close_strength"] = (
            (df["close"] - df["low"]) - (df["high"] - df["close"])
        ) / (df["high"] - df["low"] + 1e-8)

        # =========================
        # E4 Market Regime
        # 注意：这些不进入 feature_cols，只用于 prediction multiplier
        # =========================
        market = (
            df.groupby("date")
            .agg(
                market_ret_1d=("ret_1d", "mean"),
                market_volume_ratio=("volume_ratio_20d", "mean"),
                cross_sectional_dispersion=("ret_1d", "std"),
            )
            .reset_index()
            .sort_values("date")
        )

        market["market_ret_5d"] = (
            market["market_ret_1d"]
            .rolling(5, min_periods=3)
            .sum()
        )

        market["market_ret_20d"] = (
            market["market_ret_1d"]
            .rolling(20, min_periods=5)
            .sum()
        )

        market["market_vol_20d"] = (
            market["market_ret_1d"]
            .rolling(20, min_periods=5)
            .std()
        )

        df = pd.merge(df, market, how="left", on="date")

        # =========================
        # Regime Multiplier
        # =========================
        df["regime_multiplier"] = 1.0

        # 市场 20 日动量过弱：降低信号强度
        df.loc[df["market_ret_20d"] < -0.05, "regime_multiplier"] *= 0.60

        # 市场波动率过高：降低信号强度
        df.loc[df["market_vol_20d"] > 0.018, "regime_multiplier"] *= 0.75

        # 横截面离散度过高：说明市场分化/冲击较大，降低信号强度
        df.loc[df["cross_sectional_dispersion"] > 0.035, "regime_multiplier"] *= 0.75

        # 市场成交量状态极端：可能是情绪冲击，降低信号强度
        df.loc[df["market_volume_ratio"] > 1.8, "regime_multiplier"] *= 0.85

        if need_label:
            df["label"] = g["close"].shift(-LABEL_HORIZON) / df["close"] - 1

        all_num_cols = list(set(
            base_num_cols
            + feature_cols
            + [
                "market_ret_1d",
                "market_ret_5d",
                "market_ret_20d",
                "market_vol_20d",
                "market_volume_ratio",
                "cross_sectional_dispersion",
                "regime_multiplier",
                "label",
            ]
        ))

        for c in all_num_cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
                df[c] = df[c].replace([np.inf, -np.inf], np.nan)

        df["regime_multiplier"] = df["regime_multiplier"].fillna(1.0)

        # =========================
        # E5: Risk-Aware Multiplier
        # 这些变量不进入 XGBoost，只用于降低高风险状态下的 factor 强度
        # =========================

        df["risk_multiplier"] = 1.0

        # 成交量冲击过多：降低信号强度
        df.loc[df["volume_shock_count"] >= 20, "risk_multiplier"] *= 0.70
        df.loc[df["volume_shock_count"] >= 35, "risk_multiplier"] *= 0.60

        # OBI 极端持续过久：可能代表单边流动性失衡，降低信号强度
        df.loc[df["obi_pos_persistence"] >= 8, "risk_multiplier"] *= 0.85
        df.loc[df["obi_neg_persistence"] >= 8, "risk_multiplier"] *= 0.75

        # 流动性质量差：降低信号强度
        df.loc[df["avg_spread"] > df["avg_spread"].median() * 1.5, "risk_multiplier"] *= 0.85

        df["risk_multiplier"] = df["risk_multiplier"].fillna(1.0)

        return df

    def build_features(table_name, sd, ed, need_label=True):
        t0 = time.time()

        df = query_daily_bar(table_name, sd, ed, need_label=need_label)
        df = add_features(df, need_label=need_label)

        df = df[
            (df["date"] >= pd.to_datetime(sd))
            & (df["date"] <= pd.to_datetime(ed))
        ].copy()

        logger.info(
            "features built",
            experiment=EXPERIMENT,
            rows=len(df),
            elapsed=round(time.time() - t0, 2),
        )

        return df.reset_index(drop=True)

    def train_model(train_df):
        train_df["label"] = train_df["label"].replace([np.inf, -np.inf], np.nan)
        train_df = train_df.dropna(subset=["label"])

        for col in feature_cols:
            train_df[col] = train_df[col].replace([np.inf, -np.inf], np.nan)

        train_df[feature_cols] = train_df[feature_cols].fillna(0)

        model = xgb.XGBRegressor(
            n_estimators=80,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=20,
            reg_lambda=5.0,
            reg_alpha=1.0,
            tree_method="hist",
            n_jobs=-1,
            random_state=42,
        )

        model.fit(train_df[feature_cols], train_df["label"])
        return model

    def print_importance(model):
        weight_df = pd.DataFrame({
            "feature": feature_cols,
            "importance": model.feature_importances_,
        }).sort_values("importance", ascending=False)

        print("Feature importance (weight):")
        print(weight_df)

        booster = model.get_booster()
        gain = booster.get_score(importance_type="gain")

        gain_df = pd.DataFrame({
            "feature": list(gain.keys()),
            "gain": list(gain.values()),
        }).sort_values("gain", ascending=False)

        print("Feature importance by gain:")
        print(gain_df)

    # =========================
    # Train
    # =========================
    train_df = build_features(
        "bigalpha_2026_stock_bar1m",
        TRAIN_START,
        TRAIN_END,
        need_label=True,
    )

    model = train_model(train_df)
    print_importance(model)

    # =========================
    # Predict
    # =========================
    test_df = build_features(
        bar1m_table,
        start_date,
        end_date,
        need_label=False,
    )

    for col in feature_cols:
        test_df[col] = test_df[col].replace([np.inf, -np.inf], np.nan)

    test_df[feature_cols] = test_df[feature_cols].fillna(0)

    test_df["factor_raw"] = model.predict(test_df[feature_cols])

    test_df["factor"] = (
        test_df["factor_raw"]
        * test_df["risk_multiplier"].fillna(1.0)
    )

    # =========================
    # Align Stock Pool
    # =========================
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()

    stk_pool["date"] = pd.to_datetime(stk_pool["date"])
    stk_pool["instrument"] = stk_pool["instrument"].astype(str)
    test_df["instrument"] = test_df["instrument"].astype(str)

    result = pd.merge(
        stk_pool,
        test_df[["date", "instrument", "factor"]],
        how="left",
        on=["date", "instrument"],
    )

    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan)
    result["factor"] = result["factor"].fillna(0)

    return result[["date", "instrument", "factor"]]


# BigQuant的计算行
if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时自行构造数据源映射（评测时由平台注入，逻辑名固定为 "bar1m"/"financial"）
    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial'
    }
    # 本地用这段区间模拟「平台注入的测试集区间」（训练区间已在 main 内写死）
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 读取平台因子库用于回归评估，您可以换成自己的因子库
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    # 评估系统：
    # process_pools=False 表示不对因子库再做预处理（bigalpha_2026_factorlib 已处理过）
    # show=True 表示画出评估图表
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


[2026-07-09 17:25:50] [info     ] 计算因子，测试区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
[2026-07-09 17:29:34] [info     ] features built                 elapsed=223.85 experiment=E4_FILTER rows=521446
Feature importance (weight):
             feature  importance
0             ret_1d    0.139929
9     close_position    0.124962
2             ret_5d    0.109225
7   volume_ratio_20d    0.077377
11      upper_shadow    0.071353
12      lower_shadow    0.071271
8       intraday_ret    0.058545
1             ret_3d    0.057352
6            vol_20d    0.051844
4            ret_20d    0.051690
5            ret_60d    0.050890
3            ret_10d    0.050868
13    close_strength    0.042914
10         body_size    0.041778
Feature importance by gain:
             feature      gain
0             ret_1d  0.439361
9     close_position  0.392368
2             ret_5d  0.342956
7   volume_ratio_20d  0.242957
11      upper_shadow  0.224040
12      lower_shadow  0.223784
8       intraday_ret  0.183826
1